In [1]:
import numpy as np
import pandas as pd
import os

# 1. Use a raw string (r"") to handle the backslashes in Windows paths
file_path = r"C:\\Users\\Isaac\\OneDrive\\Documents\\Winter 2026 semester\\STAT 486\\tournament_model_ml.csv"

def load_data(path):
    # Check if the file actually exists at this location to debug
    if not os.path.exists(path):
        print(f"Error: The file was not found at {path}")
        return None
    return pd.read_csv(path)

# 2. Load the data
df = load_data(file_path)

if df is not None:
    # 3. Split into train (2015-2022) and test (2023-2025)
    train_data = df[(df['season'] >= 2015) & (df['season'] <= 2022)]
    test_data = df[(df['season'] >= 2023) & (df['season'] <= 2025)]

    # 4. Verify the split
    print(f"Training set: {train_data.shape[0]} rows (Seasons: {train_data['season'].unique()})")
    print(f"Testing set: {test_data.shape[0]} rows (Seasons: {test_data['season'].unique()})")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# ---> REPLACED cross_val_score WITH GridSearchCV <---
from sklearn.model_selection import GridSearchCV

X_train = train_data.drop(columns=['win_label', 'season'])
y_train = train_data['win_label']

X_test = test_data.drop(columns=['win_label', 'season'])
y_test = test_data['win_label']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Define the parameter grid and initialize base model
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'] 
}

base_model = LogisticRegression(max_iter=5000)

# 6. Set up and run GridSearchCV on the training data
grid_search = GridSearchCV(estimator=base_model, 
                           param_grid=param_grid, 
                           cv=5, 
                           scoring='accuracy', 
                           n_jobs=-1)

print("\nStarting grid search... testing all combinations...")
grid_search.fit(X_train_scaled, y_train)

# Print the best parameters found during cross-validation
print("\n--- Grid Search Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")
print("---------------------------\n")

# Extract the winning model
best_model = grid_search.best_estimator_

# 7. Make Predictions on the test set using the BEST model
y_pred = best_model.predict(X_test_scaled)

# 8. Evaluate the model performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Final Test Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

feature_names = X_train.columns

# 9. Get the coefficients from your BEST trained logistic regression model
# Notice we use best_model.coef_ now instead of model.coef_
coefficients = best_model.coef_[0]

# 10. Create a DataFrame to store feature names and their impact
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients) # Absolute value for ranking magnitude
})

# 11. Sort the variables by their importance (absolute magnitude)
importance_df = importance_df.sort_values(by='Abs_Coefficient', ascending=False)

print("\nVariable Importance (Ranked by Magnitude):")
print(importance_df[['Feature', 'Coefficient', 'Abs_Coefficient']])

# Check for eliminated variables (coefficients exactly equal to 0)
eliminated_vars = importance_df[importance_df['Coefficient'] == 0.0]

if not eliminated_vars.empty:
    print(f"\nYes! The model eliminated {len(eliminated_vars)} variables:")
    print(eliminated_vars['Feature'].tolist())
else:
    print("\nNo variables were completely eliminated (no coefficients were exactly 0.0).")

Training set: 754 rows (Seasons: [2015 2016 2017 2018 2019 2021 2022])
Testing set: 314 rows (Seasons: [2023 2024 2025])

Starting grid search... testing all combinations...

--- Grid Search Results ---
Best Parameters: {'C': 100.0, 'penalty': 'l1', 'solver': 'liblinear'}
Best Cross-Validation Accuracy: 0.7744
---------------------------

Final Test Model Accuracy: 0.7580

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.76      0.76       157
           1       0.76      0.76      0.76       157

    accuracy                           0.76       314
   macro avg       0.76      0.76      0.76       314
weighted avg       0.76      0.76      0.76       314


Confusion Matrix:
[[119  38]
 [ 38 119]]

Variable Importance (Ranked by Magnitude):
        Feature  Coefficient  Abs_Coefficient
4      SRS_diff     3.012140         3.012140
7     seed_diff     1.505788         1.505788
8  win_pct_diff     0.648465         0.648465
2   

c:\Users\Isaac\OneDrive\Documents\Winter 2026 semester\STAT 486\lab-02-Imick5555\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Isaac\OneDrive\Documents\Winter 2026 semester\STAT 486\lab-02-Imick5555\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
